# Sediment NN Evolution

Evolves a tiny neural network (72→48→32→5, ~5,237 params) to play [Sediment](https://locomot.io/sediment/) — a die-to-build platformer where your deaths become terrain.

The game is an auto-runner with tetromino-shaped players. The NN sees:
- Player state (velocity, jumps, on_ground, material, etc.)
- Current shape geometry (4×4 grid)
- Terrain lookahead (10 columns of ground/ceiling/spike/gap info)
- Nearest buzzsaw positions

And outputs one of: nothing, jump, rotate CW, rotate CCW, voluntary death.

Training uses a genetic algorithm with tournament selection and curriculum learning.

In [ ]:
# No special installs needed — just numpy (included in Colab)
import math
import random
import time
import json
import os
import multiprocessing as mp
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

print(f"CPUs available: {mp.cpu_count()}")
print(f"NumPy version: {np.__version__}")

## Game Simulator
Complete headless port of the Sediment game physics.

In [ ]:
# ── Constants ──
TILE = 12
LEVEL_H_TILES = 14
SEGMENT_W = 40
GRAVITY = 700
JUMP_FORCE = -280
RUN_SPEED = 100
WALL_SLIDE_SPEED = 40
COLUMN_CLEAR_THRESHOLD = 6

# ── Shape definitions (exact copy from JS) ──
SHAPES = {
    'I': [
        [[0,1],[1,1],[2,1],[3,1]],
        [[2,0],[2,1],[2,2],[2,3]],
        [[0,2],[1,2],[2,2],[3,2]],
        [[1,0],[1,1],[1,2],[1,3]]
    ],
    'O': [
        [[0,0],[1,0],[0,1],[1,1]],
        [[0,0],[1,0],[0,1],[1,1]],
        [[0,0],[1,0],[0,1],[1,1]],
        [[0,0],[1,0],[0,1],[1,1]]
    ],
    'T': [
        [[0,0],[1,0],[2,0],[1,1]],
        [[1,0],[1,1],[1,2],[0,1]],
        [[1,1],[0,2],[1,2],[2,2]],
        [[1,0],[1,1],[1,2],[2,1]]
    ],
    'S': [
        [[1,0],[2,0],[0,1],[1,1]],
        [[0,0],[0,1],[1,1],[1,2]],
        [[1,1],[2,1],[0,2],[1,2]],
        [[1,0],[1,1],[2,1],[2,2]]
    ],
    'Z': [
        [[0,0],[1,0],[1,1],[2,1]],
        [[2,0],[1,1],[2,1],[1,2]],
        [[0,1],[1,1],[1,2],[2,2]],
        [[1,0],[0,1],[1,1],[0,2]]
    ],
    'L': [
        [[0,0],[1,0],[2,0],[0,1]],
        [[0,0],[1,0],[1,1],[1,2]],
        [[2,1],[0,2],[1,2],[2,2]],
        [[1,0],[1,1],[1,2],[2,2]]
    ],
    'J': [
        [[0,0],[1,0],[2,0],[2,1]],
        [[1,0],[1,1],[0,2],[1,2]],
        [[0,1],[0,2],[1,2],[2,2]],
        [[1,0],[2,0],[1,1],[1,2]]
    ]
}

SHAPE_NAMES = list(SHAPES.keys())
MATERIAL_TYPES = ['solid', 'spring', 'booster']

KICK_DATA = [
    [[0,0],[-1,0],[-1,-1],[0,2],[-1,2]],
    [[0,0],[1,0],[1,1],[0,-2],[1,-2]],
    [[0,0],[1,0],[1,-1],[0,2],[1,2]],
    [[0,0],[-1,0],[-1,1],[0,-2],[-1,-2]]
]
KICK_DATA_I = [
    [[0,0],[-2,0],[1,0],[-2,1],[1,-2]],
    [[0,0],[-1,0],[2,0],[-1,-2],[2,1]],
    [[0,0],[2,0],[-1,0],[2,-1],[-1,2]],
    [[0,0],[1,0],[-2,0],[1,2],[-2,-1]]
]


def get_cells(shape_name, rotation):
    return SHAPES[shape_name][rotation]


def shape_bounds(shape_name, rotation):
    cells = get_cells(shape_name, rotation)
    min_c = min(c for c, r in cells)
    max_c = max(c for c, r in cells)
    min_r = min(r for c, r in cells)
    max_r = max(r for c, r in cells)
    return min_c, max_c, min_r, max_r, max_c - min_c + 1, max_r - min_r + 1


def pick_material(lvl):
    if lvl >= 4 and random.random() < 0.25:
        return 'spike'
    return random.choice(MATERIAL_TYPES)


def get_level_difficulty(lvl):
    return min(10, 1 + (lvl - 1) * 0.82)


def get_segments_for_level(lvl):
    return min(7, 3 + int((lvl - 1) * 0.4))

print("Game constants loaded.")

In [ ]:
class SedimentSim:
    """Headless Sediment game simulation."""

    def __init__(self, level_num=1):
        self.level_num = level_num
        self.level = []
        self.corpse_grid = []
        self.level_h = LEVEL_H_TILES
        self.level_w = 0
        self.buzzsaws = []
        self.moving_platforms = []
        self.segments = []
        self.pending_clears = []
        self.goal_x = 0
        self.time = 0.0
        self.shape_bag = []

        self.player = {
            'x': 2 * TILE, 'y': 0,
            'vx': RUN_SPEED, 'vy': 0,
            'shape': 'T', 'rotation': 0,
            'material': 'solid',
            'on_ground': False, 'on_wall': 0,
            'dead': False, 'facing': 1,
            'boost_timer': 0.0,
            'death_count': 0, 'crunch_count': 0,
            'crunch_timer': 0.0,
            'distance': 0.0, 'best_distance': 0.0,
            'spawn_x': 2 * TILE,
            'jumps_left': 2, 'coyote_timer': 0.0,
            'death_timer': 0.0, 'stuck_timer': 0.0,
        }
        self.levels_completed = 0
        self._init_level(level_num)

    def _pick_shape(self):
        if not self.shape_bag:
            self.shape_bag = list(SHAPE_NAMES)
            random.shuffle(self.shape_bag)
        return self.shape_bag.pop()

    def _init_level(self, lvl):
        self.level = []
        self.corpse_grid = []
        self.buzzsaws = []
        self.moving_platforms = []
        self.segments = []
        self.pending_clears = []
        self.level_h = LEVEL_H_TILES
        self.level_num = lvl

        num_segs = get_segments_for_level(lvl)
        for i in range(num_segs):
            self._generate_segment(i, lvl)
        self.level_w = num_segs * SEGMENT_W
        self.goal_x = (num_segs * SEGMENT_W - 4) * TILE

        p = self.player
        p['x'] = 2 * TILE
        p['y'] = (self.level_h - 3) * TILE
        p['vx'] = RUN_SPEED
        p['vy'] = 0
        p['shape'] = self._pick_shape()
        p['rotation'] = 0
        p['material'] = pick_material(lvl)
        p['on_ground'] = False
        p['on_wall'] = 0
        p['dead'] = False
        p['boost_timer'] = 0.0
        p['crunch_timer'] = 0.0
        p['distance'] = 0.0
        p['spawn_x'] = 2 * TILE
        p['jumps_left'] = 2
        p['coyote_timer'] = 0.0
        p['death_timer'] = 0.0
        p['stuck_timer'] = 0.0

    def _generate_segment(self, seg_index, lvl):
        h = self.level_h
        start_x = seg_index * SEGMENT_W
        d_base = get_level_difficulty(lvl)
        d = min(10, d_base + seg_index * 0.3)

        while len(self.level) < h:
            self.level.append([])
            self.corpse_grid.append([])

        for y in range(h):
            while len(self.level[y]) < start_x + SEGMENT_W:
                self.level[y].append(0)
                self.corpse_grid[y].append(None)
            for x in range(start_x, start_x + SEGMENT_W):
                self.level[y][x] = 1 if (y == 0 or y == h - 1) else 0
                self.corpse_grid[y][x] = None

        num_chasms = 2 + int(d * 0.8)
        chasm_min = 10 if seg_index == 0 else 3
        for _ in range(num_chasms):
            cx = start_x + chasm_min + random.randint(0, max(0, SEGMENT_W - chasm_min - 5))
            cw = 2 + int(random.random() * (1 + d * 0.4))
            for dx in range(cw):
                xx = cx + dx
                if xx < start_x + SEGMENT_W:
                    if lvl >= 8 and random.random() < min(0.8, d * 0.1):
                        if h - 2 >= 0:
                            self.level[h - 2][xx] = 2
                    else:
                        self.level[h - 1][xx] = 0

        if lvl >= 2:
            num_spikes = 1 + int(d * 0.6)
            spike_min = 10 if seg_index == 0 else 3
            for _ in range(num_spikes):
                sx = start_x + spike_min + random.randint(0, max(0, SEGMENT_W - spike_min - 4))
                sw = 2 + random.randint(0, 2)
                for dx in range(sw):
                    xx = sx + dx
                    if xx < start_x + SEGMENT_W and self.level[h - 1][xx] == 1:
                        self.level[h - 2][xx] = 2

        if lvl >= 3:
            num_cs = max(1, int(d * 0.5))
            for _ in range(num_cs):
                sx = start_x + 2 + random.randint(0, max(0, SEGMENT_W - 5))
                if sx < start_x + SEGMENT_W:
                    self.level[1][sx] = 2
                    if d > 6 and random.random() < 0.4:
                        self.level[2][sx] = 2

        if lvl >= 5:
            num_plats = 1 + int(d * 0.3)
            plat_min = 14 if seg_index == 0 else 4
            for _ in range(num_plats):
                px = start_x + plat_min + random.randint(0, max(0, SEGMENT_W - plat_min - 7))
                pw = 2 + random.randint(0, 2)
                py = 3 + random.randint(0, max(0, h - 7))
                for dx in range(pw):
                    xx = px + dx
                    if xx < start_x + SEGMENT_W and py < h:
                        self.level[py][xx] = 1

        if lvl >= 6:
            num_saws = max(1, int((d - 4) * 0.6))
            for _ in range(num_saws):
                saw_x = (start_x + 8 + random.randint(0, max(0, SEGMENT_W - 17))) * TILE
                saw_y = (2 + random.randint(0, max(0, h - 6))) * TILE
                vertical = random.random() < 0.5
                rng = (2 + random.randint(0, 2)) * TILE
                self.buzzsaws.append({
                    'x': saw_x, 'y': saw_y,
                    'base_x': saw_x, 'base_y': saw_y,
                    'r': 6, 'vertical': vertical, 'range': rng,
                    'speed': 30 + random.random() * 40,
                    'phase': random.random() * math.pi * 2,
                })

        if lvl >= 7:
            num_mov = 1 + int((d - 5) * 0.4)
            mov_min = 16 if seg_index == 0 else 5
            for _ in range(num_mov):
                mx = (start_x + mov_min + random.randint(0, max(0, SEGMENT_W - mov_min - 9))) * TILE
                my = (3 + random.randint(0, max(0, h - 7))) * TILE
                vert = lvl >= 10 and random.random() < 0.4
                self.moving_platforms.append({
                    'x': mx, 'y': my, 'w': 3,
                    'base_x': mx, 'base_y': my,
                    'vertical': vert,
                    'range': (2 + random.randint(0, 1)) * TILE,
                    'speed': 20 + random.random() * 30,
                    'phase': random.random() * math.pi * 2,
                })

        if lvl >= 9:
            num_walls = max(1, int((d - 6) * 0.5))
            for _ in range(num_walls):
                wx = start_x + 6 + random.randint(0, max(0, SEGMENT_W - 13))
                wall_h = 2 + random.randint(0, 2)
                start_y = h - 2 - wall_h
                for dy in range(wall_h):
                    yy = start_y + dy
                    if 0 <= yy < h and wx < start_x + SEGMENT_W:
                        self.level[yy][wx] = 2

        if seg_index == 0:
            for y in range(1, h - 1):
                for x in range(10):
                    if self.level[y][x] == 2:
                        self.level[y][x] = 0
            for x in range(10):
                self.level[h - 1][x] = 1

        self.segments.append(seg_index)

    def _tile_is_solid(self, tx, ty):
        if ty < 0: return True
        if ty >= self.level_h: return False
        if tx < 0 or tx >= self.level_w:
            return tx < 0
        if self.level[ty][tx] == 1: return True
        if self.corpse_grid[ty][tx] is not None: return True
        for p in self.moving_platforms:
            px1 = int(p['x'] / TILE)
            py = int(p['y'] / TILE)
            if ty == py and px1 <= tx < px1 + p['w']:
                return True
        return False

    def _tile_is_spike(self, tx, ty):
        if ty < 0 or ty >= self.level_h or tx < 0 or tx >= self.level_w:
            return False
        if self.level[ty][tx] == 2: return True
        c = self.corpse_grid[ty][tx]
        if c and c['mat'] == 'spike': return True
        return False

    def _tet_solid(self, px, py, shape, rot):
        cells = get_cells(shape, rot)
        for cx, cy in cells:
            wx = px + cx * TILE
            wy = py + cy * TILE
            l = wx // TILE
            r = (wx + TILE - 1) // TILE
            t = wy // TILE
            b = (wy + TILE - 1) // TILE
            for ty in range(t, b + 1):
                for tx in range(l, r + 1):
                    if self._tile_is_solid(tx, ty): return True
        return False

    def _tet_spike(self, px, py, shape, rot):
        cells = get_cells(shape, rot)
        for cx, cy in cells:
            wx = px + cx * TILE + 1
            wy = py + cy * TILE + 1
            l = wx // TILE
            r = (wx + TILE - 3) // TILE
            t = wy // TILE
            b = (wy + TILE - 3) // TILE
            for ty in range(t, b + 1):
                for tx in range(l, r + 1):
                    if self._tile_is_spike(tx, ty): return True
        return False

    def _corpse_mat_at(self, tx, ty):
        if 0 <= ty < self.level_h and 0 <= tx < self.level_w:
            c = self.corpse_grid[ty][tx]
            if c: return c['mat']
        return None

    def _check_adjacent_corpse(self, px, py, shape, rot, direction, mat):
        cells = get_cells(shape, rot)
        for cx, cy in cells:
            if direction == 'below':
                check_tx = (px + cx * TILE + TILE // 2) // TILE
                check_ty = (py + (cy + 1) * TILE) // TILE
            elif direction == 'above':
                check_tx = (px + cx * TILE + TILE // 2) // TILE
                check_ty = (py + cy * TILE - 1) // TILE
            elif direction == 'right':
                check_tx = (px + (cx + 1) * TILE) // TILE
                check_ty = (py + cy * TILE + TILE // 2) // TILE
            elif direction == 'left':
                check_tx = (px + cx * TILE - 1) // TILE
                check_ty = (py + cy * TILE + TILE // 2) // TILE
            else: continue
            if self._corpse_mat_at(check_tx, check_ty) == mat:
                return (check_tx, check_ty)
        return None

    def _can_escape_right(self):
        p = self.player
        shape, rot = p['shape'], p['rotation']
        max_up = 8 * TILE
        for dy in range(1, max_up + 1):
            test_y = p['y'] - dy
            if self._tet_solid(p['x'], test_y, shape, rot): break
            if not self._tet_solid(p['x'] + 1, test_y, shape, rot): return True
        for d in [1, -1]:
            new_rot = (rot + d + 4) % 4
            kicks = KICK_DATA_I if shape == 'I' else KICK_DATA
            kick_idx = rot if d == 1 else new_rot
            for kx, ky in kicks[kick_idx]:
                tx = p['x'] + (kx if d == 1 else -kx) * TILE
                ty = p['y'] + (-ky if d == 1 else ky) * TILE
                if not self._tet_solid(tx, ty, shape, new_rot):
                    if not self._tet_solid(tx + 1, ty, shape, new_rot): return True
                    for dy2 in range(1, max_up + 1):
                        jy = ty - dy2
                        if self._tet_solid(tx, jy, shape, new_rot): break
                        if not self._tet_solid(tx + 1, jy, shape, new_rot): return True
        return False

    def _try_rotate(self, direction):
        p = self.player
        old_rot = p['rotation']
        new_rot = (old_rot + direction + 4) % 4
        kicks = KICK_DATA_I if p['shape'] == 'I' else KICK_DATA
        kick_idx = old_rot if direction == 1 else new_rot
        for kx, ky in kicks[kick_idx]:
            tx = p['x'] + (kx if direction == 1 else -kx) * TILE
            ty = p['y'] + (-ky if direction == 1 else ky) * TILE
            if not self._tet_solid(tx, ty, p['shape'], new_rot):
                p['x'] = tx
                p['y'] = ty
                p['rotation'] = new_rot
                return True
        return False

    def _die(self):
        p = self.player
        if p['dead']: return
        p['dead'] = True
        p['death_timer'] = 0.5
        p['death_count'] += 1
        cells = get_cells(p['shape'], p['rotation'])
        for cx, cy in cells:
            tx = round((p['x'] + cx * TILE) / TILE)
            ty = round((p['y'] + cy * TILE) / TILE)
            if 0 <= ty < self.level_h and 0 <= tx < self.level_w:
                if p['material'] == 'spike':
                    self.level[ty][tx] = 2
                elif self.corpse_grid[ty][tx] is None and self.level[ty][tx] == 0:
                    if self.level[ty][tx] == 2:
                        self.level[ty][tx] = 0
                    self.corpse_grid[ty][tx] = {'mat': p['material']}
        self._apply_corpse_gravity()
        self._check_clears()

    def _apply_corpse_gravity(self):
        changed = True
        while changed:
            changed = False
            for y in range(self.level_h - 2, 0, -1):
                for x in range(self.level_w):
                    c = self.corpse_grid[y][x]
                    if c and not self._tile_is_solid_no_corpse(x, y + 1) and self.corpse_grid[y + 1][x] is None:
                        self.corpse_grid[y + 1][x] = c
                        self.corpse_grid[y][x] = None
                        changed = True

    def _tile_is_solid_no_corpse(self, tx, ty):
        if ty < 0: return True
        if ty >= self.level_h or tx < 0 or tx >= self.level_w: return False
        return self.level[ty][tx] == 1

    def _check_clears(self):
        max_x = len(self.segments) * SEGMENT_W
        for x in range(max_x):
            run = 0
            found = False
            for y in range(self.level_h):
                c = self.corpse_grid[y][x] if y < len(self.corpse_grid) and x < len(self.corpse_grid[y]) else None
                if c and c['mat'] != 'spike':
                    run += 1
                else:
                    if run >= COLUMN_CLEAR_THRESHOLD:
                        found = True
                        break
                    run = 0
            if run >= COLUMN_CLEAR_THRESHOLD: found = True
            if found:
                for y in range(self.level_h):
                    if y < len(self.corpse_grid) and x < len(self.corpse_grid[y]):
                        self.corpse_grid[y][x] = None
                self.player['crunch_count'] += 1
                self.player['crunch_timer'] += 3.0

        for y in range(1, self.level_h - 1):
            run = 0
            run_start = 0
            for x in range(max_x):
                c = self.corpse_grid[y][x] if x < len(self.corpse_grid[y]) else None
                if c and c['mat'] != 'spike':
                    if run == 0: run_start = x
                    run += 1
                else:
                    if run >= COLUMN_CLEAR_THRESHOLD:
                        for xx in range(run_start, run_start + run):
                            if xx < len(self.corpse_grid[y]):
                                self.corpse_grid[y][xx] = None
                        self.player['crunch_count'] += 1
                        self.player['crunch_timer'] += 3.0
                    run = 0
            if run >= COLUMN_CLEAR_THRESHOLD:
                for xx in range(run_start, run_start + run):
                    if xx < len(self.corpse_grid[y]):
                        self.corpse_grid[y][xx] = None
                self.player['crunch_count'] += 1
                self.player['crunch_timer'] += 3.0

    def _respawn(self):
        p = self.player
        p['dead'] = False
        p['x'] = p['spawn_x']
        p['y'] = (self.level_h - 3) * TILE
        p['vx'] = RUN_SPEED
        p['vy'] = 0
        p['shape'] = self._pick_shape()
        p['rotation'] = 0
        p['material'] = pick_material(self.level_num)
        p['on_ground'] = False
        p['on_wall'] = 0
        p['jumps_left'] = 2
        p['coyote_timer'] = 0
        p['stuck_timer'] = 0
        p['boost_timer'] = 0
        if self._tet_solid(p['x'], p['y'], p['shape'], p['rotation']):
            p['y'] = (self.level_h - 4) * TILE
        best_tile_x = int(p['best_distance'] / TILE)
        if best_tile_x > p['spawn_x'] // TILE + 5:
            p['spawn_x'] = max(p['spawn_x'], (best_tile_x - 5) * TILE)

    def step(self, dt, action=0):
        """Advance one frame. action: 0=nothing, 1=jump, 2=rotate_cw, 3=rotate_ccw, 4=voluntary_death"""
        p = self.player
        self.time += dt

        for saw in self.buzzsaws:
            t = self.time * saw['speed'] / saw['range'] + saw['phase']
            if saw['vertical']: saw['y'] = saw['base_y'] + math.sin(t) * saw['range']
            else: saw['x'] = saw['base_x'] + math.sin(t) * saw['range']

        for mp in self.moving_platforms:
            t = self.time * mp['speed'] / mp['range'] + mp['phase']
            if mp['vertical']: mp['y'] = mp['base_y'] + math.sin(t) * mp['range']
            else: mp['x'] = mp['base_x'] + math.sin(t) * mp['range']

        if p['dead']:
            p['death_timer'] -= dt
            if p['death_timer'] <= 0: self._respawn()
            return

        jump_buffered = False
        if action == 1: jump_buffered = True
        elif action == 2: self._try_rotate(1)
        elif action == 3: self._try_rotate(-1)
        elif action == 4:
            self._die()
            return

        if p['boost_timer'] > 0: p['boost_timer'] -= dt
        else: p['vx'] = RUN_SPEED

        p['vy'] += GRAVITY * dt
        if p['vy'] > 400: p['vy'] = 400

        bounds = shape_bounds(p['shape'], p['rotation'])
        right_col = bounds[1]
        p['on_wall'] = 0
        for cy in range(bounds[2], bounds[3] + 1):
            check_tx = (p['x'] + (right_col + 1) * TILE) // TILE
            check_ty = (p['y'] + cy * TILE + TILE // 2) // TILE
            if self._tile_is_solid(check_tx, check_ty):
                p['on_wall'] = 1
                break

        if p['on_wall'] and p['vy'] > WALL_SLIDE_SPEED:
            p['vy'] = WALL_SLIDE_SPEED

        if p['on_ground']: p['coyote_timer'] = 0.06
        elif p['coyote_timer'] > 0: p['coyote_timer'] -= dt

        if jump_buffered:
            if p['on_wall']:
                p['vy'] = JUMP_FORCE * 0.9
                p['vx'] = -180 if p['on_wall'] == 1 else 180
                p['jumps_left'] = 1
                p['boost_timer'] = 0.15
                p['on_wall'] = 0
            elif p['on_ground'] or p['coyote_timer'] > 0:
                p['vy'] = JUMP_FORCE
                p['jumps_left'] = 1
                p['coyote_timer'] = 0
            elif p['jumps_left'] > 0:
                p['vy'] = JUMP_FORCE * 0.7
                p['jumps_left'] -= 1

        dx = p['vx'] * dt
        step_dir = 1 if dx > 0 else -1
        hit_wall_x = False
        for _ in range(int(abs(dx))):
            if not self._tet_solid(p['x'] + step_dir, p['y'], p['shape'], p['rotation']):
                p['x'] += step_dir
            else:
                p['vx'] = 0
                hit_wall_x = True
                break

        if hit_wall_x:
            adj = self._check_adjacent_corpse(p['x'], p['y'], p['shape'], p['rotation'], 'right', 'booster')
            if adj:
                p['vx'] = 350
                p['vy'] = -140
                p['boost_timer'] = 0.25

        dy = p['vy'] * dt
        step_dir_y = 1 if dy > 0 else -1
        p['on_ground'] = False
        for _ in range(int(abs(dy))):
            if not self._tet_solid(p['x'], p['y'] + step_dir_y, p['shape'], p['rotation']):
                p['y'] += step_dir_y
            else:
                if step_dir_y > 0:
                    p['on_ground'] = True
                    p['jumps_left'] = 2
                p['vy'] = 0
                break

        if p['on_ground']:
            spring = self._check_adjacent_corpse(p['x'], p['y'], p['shape'], p['rotation'], 'below', 'spring')
            if spring:
                p['vy'] = JUMP_FORCE * 1.5
                p['on_ground'] = False
            booster = self._check_adjacent_corpse(p['x'], p['y'], p['shape'], p['rotation'], 'below', 'booster')
            if booster:
                p['vx'] = p['facing'] * 350
                p['vy'] = JUMP_FORCE * 0.3
                p['on_ground'] = False
                p['boost_timer'] = 0.25

        if p['vy'] == 0 and step_dir_y < 0:
            spring_above = self._check_adjacent_corpse(p['x'], p['y'], p['shape'], p['rotation'], 'above', 'spring')
            if spring_above: p['vy'] = abs(JUMP_FORCE) * 1.2

        if p['y'] > self.level_h * TILE + 20:
            if p['crunch_timer'] > 0:
                p['vy'] = JUMP_FORCE
                p['y'] = (self.level_h - 2) * TILE
            else:
                self._die()
                return

        if hit_wall_x and p['vx'] >= 0:
            if not self._can_escape_right():
                p['stuck_timer'] += dt
                if p['stuck_timer'] >= 0.5:
                    self._die()
                    return
            else: p['stuck_timer'] = 0
        else: p['stuck_timer'] = 0

        if p['crunch_timer'] > 0: p['crunch_timer'] -= dt
        else:
            if self._tet_spike(p['x'], p['y'], p['shape'], p['rotation']):
                self._die()
                return
            cells = get_cells(p['shape'], p['rotation'])
            for cx, cy in cells:
                cell_cx = p['x'] + cx * TILE + TILE / 2
                cell_cy = p['y'] + cy * TILE + TILE / 2
                for saw in self.buzzsaws:
                    dist = math.sqrt((cell_cx - saw['x'])**2 + (cell_cy - saw['y'])**2)
                    if dist < saw['r'] + TILE / 2:
                        self._die()
                        return

        cells = get_cells(p['shape'], p['rotation'])
        max_cell_x = max(p['x'] + cx * TILE for cx, cy in cells)
        p['distance'] = max(p['distance'], max_cell_x)
        p['best_distance'] = max(p['best_distance'], p['distance'])

        if max_cell_x >= self.goal_x:
            self.levels_completed += 1
            self._init_level(self.level_num + 1)

    def get_nn_inputs(self):
        """Extract the 72-dimensional input vector for the NN."""
        p = self.player
        if p['dead']: return np.zeros(72, dtype=np.float32)

        inputs = []
        inputs.append(p['vy'] / 400.0)
        inputs.append(p['jumps_left'] / 2.0)
        inputs.append(1.0 if p['on_ground'] else 0.0)
        inputs.append((p['on_wall'] + 1) / 2.0)
        inputs.append(p['y'] / (self.level_h * TILE))
        inputs.append(1.0 if p['boost_timer'] > 0 else 0.0)
        inputs.append(1.0 if p['crunch_timer'] > 0 else 0.0)
        inputs.append(min(1.0, p['stuck_timer'] / 0.5))
        inputs.append(min(1.0, p['distance'] / max(1, self.goal_x)))

        cells = get_cells(p['shape'], p['rotation'])
        min_c = min(c for c, r in cells)
        min_r = min(r for c, r in cells)
        grid_4x4 = [0.0] * 16
        for c, r in cells:
            gc = c - min_c
            gr = r - min_r
            if 0 <= gc < 4 and 0 <= gr < 4:
                grid_4x4[gr * 4 + gc] = 1.0
        inputs.extend(grid_4x4)

        inputs.append(1.0 if p['material'] == 'spring' else 0.0)
        inputs.append(1.0 if p['material'] == 'booster' else 0.0)
        inputs.append(1.0 if p['material'] == 'spike' else 0.0)

        player_col = int(p['x'] / TILE)
        for col_offset in range(10):
            col = player_col + col_offset + 1
            ground_h = 0.0
            ceiling_h = 0.0
            has_spike = 0.0
            has_gap = 1.0
            for y in range(self.level_h):
                solid = False
                if 0 <= col < self.level_w:
                    if self.level[y][col] == 1: solid = True
                    elif self.level[y][col] == 2: has_spike = 1.0
                    c = self.corpse_grid[y][col] if col < len(self.corpse_grid[y]) else None
                    if c:
                        solid = True
                        if c['mat'] == 'spike': has_spike = 1.0
                if solid and ceiling_h == 0.0 and y > 0:
                    ceiling_h = y / self.level_h
            for y in range(self.level_h - 1, -1, -1):
                if 0 <= col < self.level_w:
                    if self.level[y][col] == 1 or self.corpse_grid[y][col] is not None:
                        ground_h = (self.level_h - y) / self.level_h
                        has_gap = 0.0
                        break
            inputs.extend([ground_h, ceiling_h, has_spike, has_gap])

        player_cx = p['x'] + TILE * 1.5
        player_cy = p['y'] + TILE * 1.5
        saw_dists = []
        for saw in self.buzzsaws:
            dx_s = (saw['x'] - player_cx) / (10 * TILE)
            dy_s = (saw['y'] - player_cy) / (self.level_h * TILE)
            dist = dx_s**2 + dy_s**2
            saw_dists.append((dist, dx_s, dy_s))
        saw_dists.sort()
        for i in range(2):
            if i < len(saw_dists):
                inputs.append(max(-1, min(1, saw_dists[i][1])))
                inputs.append(max(-1, min(1, saw_dists[i][2])))
            else:
                inputs.extend([0.0, 0.0])

        return np.array(inputs, dtype=np.float32)


print("SedimentSim loaded.")

# Quick test
sim = SedimentSim(level_num=1)
print(f"Level 1: {len(sim.segments)} segs, goal at {sim.goal_x}px")
print(f"NN input dims: {sim.get_nn_inputs().shape[0]}")

## Neural Network

In [ ]:
class SmallNN:
    """Tiny feedforward NN: 72 -> 48 -> 32 -> 5"""
    LAYER_SIZES = [72, 48, 32, 5]

    def __init__(self, weights=None):
        if weights is None:
            weights = self.random_weights()
        self.w1, self.b1, self.w2, self.b2, self.w3, self.b3 = self._unpack(weights)

    @staticmethod
    def num_params():
        sizes = SmallNN.LAYER_SIZES
        total = 0
        for i in range(len(sizes) - 1):
            total += sizes[i] * sizes[i+1] + sizes[i+1]
        return total

    @staticmethod
    def random_weights():
        return np.random.randn(SmallNN.num_params()).astype(np.float32) * 0.5

    def _unpack(self, flat):
        sizes = self.LAYER_SIZES
        idx = 0
        params = []
        for i in range(len(sizes) - 1):
            n_in, n_out = sizes[i], sizes[i+1]
            w = flat[idx:idx + n_in * n_out].reshape(n_out, n_in)
            idx += n_in * n_out
            b = flat[idx:idx + n_out]
            idx += n_out
            params.extend([w, b])
        return params

    def forward(self, x):
        h = np.maximum(0, self.w1 @ x + self.b1)
        h = np.maximum(0, self.w2 @ h + self.b2)
        logits = self.w3 @ h + self.b3
        return int(np.argmax(logits))

    def to_json(self):
        return json.dumps({
            'w1': self.w1.tolist(), 'b1': self.b1.tolist(),
            'w2': self.w2.tolist(), 'b2': self.b2.tolist(),
            'w3': self.w3.tolist(), 'b3': self.b3.tolist(),
        })


def evaluate_agent(weights, level_num=1, max_time=60.0, dt=1/60):
    """Run a single agent evaluation. Returns fitness."""
    sim = SedimentSim(level_num)
    nn = SmallNN(weights)
    while sim.time < max_time:
        if sim.player['dead']:
            sim.step(dt, action=0)
            continue
        inputs = sim.get_nn_inputs()
        action = nn.forward(inputs)
        sim.step(dt, action)
    p = sim.player
    fitness = p['best_distance'] + 500 * sim.levels_completed + 50 * p['crunch_count']
    return fitness


def evaluate_with_seed(args):
    """Worker function for parallel evaluation."""
    weights, level_num, eval_time, seed = args
    random.seed(seed)
    np.random.seed(seed)
    return evaluate_agent(weights, level_num=level_num, max_time=eval_time)


print(f"NN params: {SmallNN.num_params()}")

# Quick fitness test
w = SmallNN.random_weights()
f = evaluate_agent(w, level_num=1, max_time=5.0)
print(f"Random agent fitness (5s): {f:.0f}")

## Evolutionary Training

In [ ]:
class Evolver:
    """Genetic algorithm for evolving Sediment NN agents."""

    def __init__(self, pop_size=200, level_num=1, eval_time=30.0,
                 elite_frac=0.1, tournament_size=7,
                 mutation_rate=0.15, mutation_scale=0.3,
                 crossover_rate=0.7, num_evals=3, workers=None):
        self.pop_size = pop_size
        self.num_params = SmallNN.num_params()
        self.level_num = level_num
        self.eval_time = eval_time
        self.elite_frac = elite_frac
        self.tournament_size = tournament_size
        self.mutation_rate = mutation_rate
        self.mutation_scale = mutation_scale
        self.crossover_rate = crossover_rate
        self.num_evals = num_evals
        self.workers = workers or max(1, mp.cpu_count() - 1)

        self.population = [SmallNN.random_weights() for _ in range(pop_size)]
        self.fitnesses = np.zeros(pop_size)
        self.generation = 0
        self.best_fitness = 0.0
        self.best_weights = None
        self.history = []
        self.levels_beaten = set()

    def evaluate_population(self):
        gen_seed_base = self.generation * 10000
        all_tasks = []
        for i, w in enumerate(self.population):
            for e in range(self.num_evals):
                seed = gen_seed_base + e * 1000 + i
                all_tasks.append((w, self.level_num, self.eval_time, seed))

        with mp.Pool(self.workers) as pool:
            all_results = pool.map(evaluate_with_seed, all_tasks)

        for i in range(self.pop_size):
            agent_fits = all_results[i * self.num_evals : (i + 1) * self.num_evals]
            self.fitnesses[i] = np.mean(agent_fits)

        best_idx = np.argmax(self.fitnesses)
        if self.fitnesses[best_idx] > self.best_fitness:
            self.best_fitness = self.fitnesses[best_idx]
            self.best_weights = self.population[best_idx].copy()

    def _tournament_select(self):
        indices = np.random.randint(0, self.pop_size, size=self.tournament_size)
        best = indices[np.argmax(self.fitnesses[indices])]
        return self.population[best]

    def _crossover(self, parent_a, parent_b):
        mask = np.random.random(self.num_params) < 0.5
        return np.where(mask, parent_a, parent_b)

    def _mutate(self, weights):
        mask = np.random.random(self.num_params) < self.mutation_rate
        noise = np.random.randn(self.num_params).astype(np.float32) * self.mutation_scale
        weights[mask] += noise[mask]
        return weights

    def evolve_generation(self):
        sorted_indices = np.argsort(self.fitnesses)[::-1]
        elite_count = max(2, int(self.pop_size * self.elite_frac))
        new_pop = []
        for i in range(elite_count):
            new_pop.append(self.population[sorted_indices[i]].copy())
        while len(new_pop) < self.pop_size:
            parent_a = self._tournament_select()
            if np.random.random() < self.crossover_rate:
                parent_b = self._tournament_select()
                child = self._crossover(parent_a, parent_b)
            else:
                child = parent_a.copy()
            child = self._mutate(child)
            new_pop.append(child)
        self.population = new_pop
        self.generation += 1

    def check_curriculum(self):
        goal_x = (get_segments_for_level(self.level_num) * SEGMENT_W - 4) * TILE
        median_fit = np.median(self.fitnesses)
        best_fit = np.max(self.fitnesses)
        if best_fit > goal_x and median_fit > goal_x * 0.6:
            self.levels_beaten.add(self.level_num)
            self.level_num += 1
            return True
        return False

    def save_checkpoint(self, path):
        np.savez_compressed(path,
            population=np.array(self.population),
            fitnesses=self.fitnesses,
            best_weights=self.best_weights if self.best_weights is not None else np.zeros(1),
            best_fitness=self.best_fitness,
            generation=self.generation,
            level_num=self.level_num,
            history=np.array(self.history) if self.history else np.zeros((0, 5)),
        )

    def load_checkpoint(self, path):
        data = np.load(path, allow_pickle=True)
        self.population = list(data['population'])
        self.fitnesses = data['fitnesses']
        self.best_weights = data['best_weights']
        self.best_fitness = float(data['best_fitness'])
        self.generation = int(data['generation'])
        self.level_num = int(data['level_num'])
        if 'history' in data and data['history'].shape[0] > 0:
            self.history = data['history'].tolist()

    def export_best(self, path):
        if self.best_weights is None:
            print("No best weights to export!")
            return
        nn = SmallNN(self.best_weights)
        with open(path, 'w') as f:
            f.write(nn.to_json())
        print(f"Exported best weights to {path}")


print("Evolver loaded.")

## Train!

Adjust parameters below. On Colab (2 CPUs), ~200 pop takes ~5-10s per generation.

Training auto-advances to harder levels when agents consistently beat the current one.

In [ ]:
# ── Training parameters ──
POP_SIZE = 200
GENERATIONS = 500
STARTING_LEVEL = 1
EVAL_TIME = 30.0    # game seconds per evaluation
NUM_EVALS = 2       # average over N runs per agent
CHECKPOINT_EVERY = 25

# Create evolver
evolver = Evolver(
    pop_size=POP_SIZE,
    level_num=STARTING_LEVEL,
    eval_time=EVAL_TIME,
    num_evals=NUM_EVALS,
)

# Optional: resume from checkpoint
# evolver.load_checkpoint('best.npz')

print(f"Population: {POP_SIZE} | Params: {evolver.num_params} | Workers: {evolver.workers}")
print(f"Starting level: {STARTING_LEVEL} | Eval time: {EVAL_TIME}s")

In [ ]:
# ── Training loop with live plotting ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for gen in range(GENERATIONS):
    t0 = time.time()

    evolver.evaluate_population()

    best = np.max(evolver.fitnesses)
    mean = np.mean(evolver.fitnesses)
    median = np.median(evolver.fitnesses)
    evolver.history.append([evolver.generation, best, mean, median, evolver.level_num])

    elapsed = time.time() - t0
    goal_x = (get_segments_for_level(evolver.level_num) * SEGMENT_W - 4) * TILE
    pct = min(100, best / goal_x * 100)

    # Curriculum check
    advanced = evolver.check_curriculum()

    # Print progress
    status = f"Gen {evolver.generation:4d} | L{evolver.level_num} | {pct:5.1f}% | "
    status += f"best:{best:7.0f} mean:{mean:7.0f} med:{median:7.0f} | {elapsed:.1f}s"
    if advanced:
        status += f" >>> LEVEL UP!"
    print(status)

    # Checkpoint
    if (evolver.generation + 1) % CHECKPOINT_EVERY == 0 or advanced:
        evolver.save_checkpoint('best.npz')
        evolver.export_best('best_weights.json')

    # Live plot every 10 generations
    if (evolver.generation + 1) % 10 == 0 and evolver.history:
        clear_output(wait=True)
        h = np.array(evolver.history)

        ax1.clear()
        ax1.plot(h[:, 0], h[:, 1], 'r-', label='Best', linewidth=2)
        ax1.plot(h[:, 0], h[:, 2], 'b-', label='Mean', alpha=0.7)
        ax1.plot(h[:, 0], h[:, 3], 'g--', label='Median', alpha=0.7)
        ax1.axhline(y=goal_x, color='gold', linestyle=':', label=f'Goal (L{evolver.level_num})', alpha=0.8)
        ax1.set_xlabel('Generation')
        ax1.set_ylabel('Fitness')
        ax1.set_title('Fitness Over Generations')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.clear()
        ax2.hist(evolver.fitnesses, bins=30, color='#68BBC4', edgecolor='#2A474F')
        ax2.axvline(x=goal_x, color='gold', linestyle=':', label=f'Goal', linewidth=2)
        ax2.set_xlabel('Fitness')
        ax2.set_ylabel('Count')
        ax2.set_title(f'Gen {evolver.generation} Distribution')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        display(fig)
        print(status)

    evolver.evolve_generation()

# Final save
evolver.save_checkpoint('best.npz')
evolver.export_best('best_weights.json')
print(f"\nDone! Best fitness: {evolver.best_fitness:.0f}")
print(f"Levels beaten: {sorted(evolver.levels_beaten) if evolver.levels_beaten else 'none yet'}")

## Export Best Agent

Download `best_weights.json` to use in the browser version.

In [ ]:
# Export final weights
evolver.export_best('best_weights.json')

# Show weight stats
if evolver.best_weights is not None:
    w = evolver.best_weights
    print(f"Weight stats: min={w.min():.3f}, max={w.max():.3f}, "
          f"mean={w.mean():.3f}, std={w.std():.3f}")
    print(f"File size: {os.path.getsize('best_weights.json') / 1024:.1f} KB")

# Download link (Colab)
try:
    from google.colab import files
    files.download('best_weights.json')
    files.download('best.npz')
except ImportError:
    print("Not on Colab — files saved locally.")

## Watch the Best Agent Play

ASCII visualization of the best agent's run.

In [ ]:
def visualize_run(weights, level_num=1, max_time=20.0, dt=1/60, snapshot_interval=2.0):
    """ASCII visualization of an agent's run."""
    random.seed(42)
    np.random.seed(42)
    sim = SedimentSim(level_num)
    nn = SmallNN(weights)

    snapshots = []
    next_snapshot = 0.0
    action_names = ['·', 'J', 'R', 'L', 'D']
    actions_taken = []

    while sim.time < max_time:
        if sim.player['dead']:
            sim.step(dt, action=0)
            continue

        inputs = sim.get_nn_inputs()
        action = nn.forward(inputs)
        actions_taken.append(action)

        if sim.time >= next_snapshot:
            p = sim.player
            # Build ASCII view around player
            view_w = 30
            player_col = int(p['x'] / TILE)
            player_row = int(p['y'] / TILE)
            start_col = max(0, player_col - 5)

            lines = []
            lines.append(f"  t={sim.time:.1f}s  pos=({p['x']:.0f},{p['y']:.0f})  "
                        f"deaths={p['death_count']}  dist={p['best_distance']:.0f}  "
                        f"action={action_names[action]}")

            for y in range(sim.level_h):
                row = ''
                for x in range(start_col, min(start_col + view_w, sim.level_w)):
                    # Check if player is here
                    is_player = False
                    if not p['dead']:
                        cells = get_cells(p['shape'], p['rotation'])
                        for cx, cy in cells:
                            px_tile = round((p['x'] + cx * TILE) / TILE)
                            py_tile = round((p['y'] + cy * TILE) / TILE)
                            if px_tile == x and py_tile == y:
                                is_player = True
                                break

                    if is_player:
                        row += '@'
                    elif sim.level[y][x] == 2:
                        row += '^'  # spike
                    elif sim.level[y][x] == 1:
                        row += '#'  # wall
                    elif sim.corpse_grid[y][x] is not None:
                        m = sim.corpse_grid[y][x]['mat']
                        row += {'solid': 'o', 'spring': 's', 'booster': 'b', 'spike': '^'}.get(m, 'o')
                    else:
                        row += '.'
                lines.append(f"  {row}")

            snapshots.append('\n'.join(lines))
            next_snapshot += snapshot_interval

        sim.step(dt, action)

    # Print snapshots
    for s in snapshots:
        print(s)
        print()

    # Action distribution
    if actions_taken:
        from collections import Counter
        counts = Counter(actions_taken)
        total = len(actions_taken)
        print("Action distribution:")
        for a, name in enumerate(action_names):
            pct = counts.get(a, 0) / total * 100
            print(f"  {name}: {pct:.1f}%")

    p = sim.player
    print(f"\nFinal: dist={p['best_distance']:.0f} deaths={p['death_count']} "
          f"crunches={p['crunch_count']} levels={sim.levels_completed}")


# Watch the best agent
if evolver.best_weights is not None:
    visualize_run(evolver.best_weights, level_num=1, max_time=20.0)
else:
    print("No trained agent yet — run training first.")